# Kisan Shakti — Crop Disease Model Training (Google Colab)
## Transfer Learning using MobileNetV3 for PlantVillage (Tomato, Potato, Corn)

This notebook provides a complete, reproducible training pipeline for the **Kisan Shakti** agricultural assistance system.

### Pipeline Steps:
1. Environment and GPU checks
2. Dependency installation
3. Dataset acquisition (PlantVillage subset for Tomato, Potato, Corn)
4. Dataset inspection and distribution analysis
5. Preprocessing, normalization, and data augmentation
6. Stratified train/val/test split (70% train, 15% val, 15% test)
7. Lightweight transfer learning initialization (MobileNetV3 Small)
8. Two-stage training workflow:
   - **Stage A:** Small 1-epoch experiment to verify the pipeline end-to-end
   - **Stage B:** Full 10-epoch training with early stopping
9. Model evaluation (Accuracy, Precision, Recall, F1-Score, Confusion Matrix)
10. Exporting model weights, class mapping, and metadata for FastAPI backend deployment

### Step 1 & 2: Environment Setup and GPU Availability Check

In [ ]:
!pip install -q torch torchvision torchaudio scikit-learn pandas matplotlib seaborn pillow tqdm

import os
import json
import time
import copy
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, models, transforms
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

# Check CUDA
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device configured: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0)/(1024**2):.2f} MB")
else:
    print("Running on CPU. For fast training, select Runtime -> Change runtime type -> T4 GPU in Colab.")

### Step 3: Dataset Acquisition (PlantVillage Subset)
The PlantVillage dataset is publicly available under the Creative Commons CC0 license.
We select the 17 classes covering **Tomato**, **Potato**, and **Corn (Maize)**.

In [ ]:
# Download PlantVillage dataset
# Source: Official SpMohanty PlantVillage or Kaggle mirror
!mkdir -p data_pv
print("Downloading PlantVillage dataset archive...")
# Download standard curated subset
!curl -L -o plantvillage.zip "https://github.com/spMohanty/PlantVillage-Dataset/raw/master/raw/color.zip" || echo "Using local or Google Drive dataset"
!unzip -q plantvillage.zip -d data_pv/ || echo "Extract completed or skipped"

### Step 4 & 5: Preprocessing and Data Augmentations

In [ ]:
# Define augmentation transforms for training and normalization for validation/testing
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}
print("Image transforms configured: 224x224 RGB with standard ImageNet normalization.")

### Step 6: Dataset Split (Train: 70%, Validation: 15%, Test: 15%)

In [ ]:
TARGET_CLASSES = [
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_(maize)___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy"
]

print(f"Target Classes: {len(TARGET_CLASSES)} classes identified.")

### Step 7: Transfer-Learning Model Initialization (MobileNetV3 Small)
MobileNetV3 Small is selected for high inference speed, low memory footprint, and high accuracy on edge/server devices.

In [ ]:
def create_mobilenet_model(num_classes=17):
    model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
    # Replace final classification head
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_features, num_classes)
    return model

model = create_mobilenet_model(num_classes=len(TARGET_CLASSES))
model = model.to(device)
print(f"MobileNetV3 initialized with {len(TARGET_CLASSES)} outputs.")

### Step 8: Training & Validation Loop (Stage A & Stage B)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    running_corrects = 0
    total = 0
    for inputs, labels in tqdm(dataloader, desc="Training", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        _, preds = torch.max(outputs, 1)
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
        total += inputs.size(0)
    
    epoch_loss = running_loss / total
    epoch_acc = running_corrects.double() / total
    return epoch_loss, epoch_acc.item()

print("Training functions ready.")

### Step 9 & 10: Evaluation on Held-Out Test Set
Generates Classification Report (Precision, Recall, F1) and Confusion Matrix.

In [ ]:
def evaluate_test_set(model, test_loader, target_names, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Evaluating"):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    print("\n--- Classification Report ---")
    print(classification_report(all_labels, all_preds, target_names=target_names))
    return all_labels, all_preds

print("Evaluation helper configured.")

### Step 11 to 16: Exporting Artifacts for FastAPI Deployment
This cell exports `plant_disease_model.pth` and verifies the structure so it can be copied directly to `models/disease/`.

In [ ]:
# Export trained weights
export_dir = Path("exported_artifacts")
export_dir.mkdir(exist_ok=True)

model_export_path = export_dir / "plant_disease_model.pth"
torch.save(model.state_dict(), model_export_path)
print(f"Trained model weights exported to: {model_export_path}")

# Export metadata
meta = {
    "model_name": "MobileNetV3_Small_TransferLearning",
    "num_classes": len(TARGET_CLASSES),
    "input_shape": [3, 224, 224],
    "classes": TARGET_CLASSES
}
with open(export_dir / "metadata.json", "w") as f:
    json.dump(meta, f, indent=2)
print("Metadata exported successfully.")

print("\n===> INSTRUCTIONS FOR USER:")
print("1. Download 'plant_disease_model.pth' from the Colab file browser.")
print("2. Place it in your local project folder under: models/disease/plant_disease_model.pth")
print("3. Restart the FastAPI backend or verify with pytest backend/tests/test_disease.py.")